# 双位点吸附模型构建工具（NEB 最短路径设计）

本工具用于创建**两个**吸附物种在 Cu(100) 表面的双位点吸附模型，**确保原子以最短距离到达高对称位点**，减少 NEB 插值过程中的扭转和多余位移。

## 最短路径设计原则

1. **位点对选择**：同类位点时，B 取**最近邻**高对称位点（如 hollow-hollow 用相邻 hollow，距离 a/4≈1.8 Å），而非对角（2.6 Å），使反应路径更短
2. **固定原子顺序**：`Cu → A物种原子 → B物种原子`，确保 ini/fin 一一对应
3. **NEB 前原子对齐**：用 `align_fin_for_neb()` 重排 fin 的吸附原子，使每个原子对应 ini 中位移最小的目标，实现最短路径插值

## 支持的物种组合
- A/B 可为：H, O, OH, H2, H2O（单原子或分子）
- 位点对：hollow-hollow, hollow-top, hollow-bridge, top-bridge 等

## 第一步：全局设置（复用单吸附的设置）

In [ ]:
import numpy as np
from ase import Atoms
from ase.io import read, write
from pathlib import Path

# ==========================================================
# 全局设置（与单吸附模型构建.ipynb 一致）
# ==========================================================

surface_poscar = "/home/ucaqzh0/thermol/100_water/absorption/absorption/POSCAR"
base_output_dir = Path("./absorption_double_models")
base_output_dir.mkdir(exist_ok=True)

print("正在读取表面结构...")
surface_atoms = read(surface_poscar, format='vasp')
print(f"✓ 已读取表面结构: {len(surface_atoms)} 原子")
print(f"✓ 表面结构文件: {surface_poscar}")

## 第二步：核心函数定义（含双位点逻辑）

**重要**：需先运行 `单吸附模型构建.ipynb` 中的核心函数 cell，或复制以下完整定义。

In [ ]:
# ==========================================================
# 从单吸附复用：位点坐标、分子创建、旋转
# ==========================================================

def get_surface_site_position(surface_atoms, site='hollow'):
    """单吸附的位点计算（确保与单吸附一致）"""
    cell = surface_atoms.get_cell()
    a = np.linalg.norm(cell[0])
    b = np.linalg.norm(cell[1])
    center_xy = np.array([a/2, b/2])
    cu_indices = [i for i, s in enumerate(surface_atoms.get_chemical_symbols()) if s.upper() == 'CU']
    if not cu_indices:
        raise ValueError("未找到Cu原子")
    cu_positions = surface_atoms.get_positions()[cu_indices]
    top_z = np.max(cu_positions[:, 2])
    top_cu_positions = cu_positions[cu_positions[:, 2] > top_z - 0.1]
    
    if site == 'hollow':
        site_pos = np.array([center_xy[0], center_xy[1], top_z])
    elif site == 'top':
        distances_to_center = np.linalg.norm(top_cu_positions[:, :2] - center_xy, axis=1)
        min_distance = np.min(distances_to_center)
        candidates = np.where(distances_to_center < min_distance + 1e-6)[0]
        if len(candidates) > 1:
            candidate_positions = top_cu_positions[candidates]
            xy_sum = candidate_positions[:, 0] + candidate_positions[:, 1]
            nearest_idx = candidates[np.argmax(xy_sum)]
        else:
            nearest_idx = candidates[0]
        site_pos = top_cu_positions[nearest_idx].copy()
        site_pos[2] = top_z
    elif site == 'bridge':
        distances_to_center = np.linalg.norm(top_cu_positions[:, :2] - center_xy, axis=1)
        min_distance = np.min(distances_to_center)
        candidates = np.where(distances_to_center < min_distance + 1e-6)[0]
        if len(candidates) > 1:
            candidate_positions = top_cu_positions[candidates]
            xy_sum = candidate_positions[:, 0] + candidate_positions[:, 1]
            nearest_idx = candidates[np.argmax(xy_sum)]
        else:
            nearest_idx = candidates[0]
        p1 = top_cu_positions[nearest_idx]
        distances_to_p1 = np.linalg.norm(top_cu_positions[:, :2] - p1[:2], axis=1)
        distances_to_p1[nearest_idx] = np.inf
        neighbor_distances = np.abs(distances_to_p1 - a/2)
        min_neighbor_distance = np.min(neighbor_distances)
        if min_neighbor_distance < 0.3:
            neighbor_candidates = np.where(neighbor_distances < min_neighbor_distance + 1e-6)[0]
            if len(neighbor_candidates) > 1:
                neighbor_positions = top_cu_positions[neighbor_candidates]
                xy_sum_neighbors = neighbor_positions[:, 0] + neighbor_positions[:, 1]
                neighbor_idx = neighbor_candidates[np.argmax(xy_sum_neighbors)]
            else:
                neighbor_idx = neighbor_candidates[0]
            p2 = top_cu_positions[neighbor_idx]
            site_pos = (p1 + p2) / 2
            site_pos[2] = top_z
        else:
            bridge_candidates = [
                np.array([center_xy[0], center_xy[1] - a/4, top_z]),
                np.array([center_xy[0] - a/4, center_xy[1], top_z]),
                np.array([center_xy[0], center_xy[1] + a/4, top_z]),
                np.array([center_xy[0] + a/4, center_xy[1], top_z]),
            ]
            xy_sums = [cand[0] + cand[1] for cand in bridge_candidates]
            site_pos = bridge_candidates[np.argmax(xy_sums)]
    else:
        raise ValueError(f"未知的位点类型: {site}")
    return site_pos


def create_molecule(molecule_type):
    """创建分子（与单吸附一致）"""
    if molecule_type == 'H':
        molecule = Atoms('H', positions=[[0, 0, 0]])
        anchor_idx = 0
    elif molecule_type == 'O':
        molecule = Atoms('O', positions=[[0, 0, 0]])
        anchor_idx = 0
    elif molecule_type == 'OH':
        molecule = Atoms('OH', positions=[[0, 0, 0], [0, 0, 0.97]])
        anchor_idx = 0
    elif molecule_type == 'H2':
        molecule = Atoms('HH', positions=[[0, 0, 0], [0.74, 0, 0]])
        anchor_idx = 0
    elif molecule_type == 'H2O':
        oh_length = 0.96
        hoh_angle_rad = 104.5 * np.pi / 180
        half_angle = hoh_angle_rad / 2
        o_pos = np.array([0, 0, 0])
        h1_pos = np.array([oh_length * np.cos(half_angle), oh_length * np.sin(half_angle), 0])
        h2_pos = np.array([oh_length * np.cos(half_angle), -oh_length * np.sin(half_angle), 0])
        molecule = Atoms('OHH', positions=[o_pos, h1_pos, h2_pos])
        anchor_idx = 0
    else:
        raise ValueError(f"不支持的分子类型: {molecule_type}")
    return molecule, anchor_idx


def rotation_matrix_from_vectors(v1, v2):
    v1 = np.array(v1, dtype=float) / np.linalg.norm(v1)
    v2 = np.array(v2, dtype=float) / np.linalg.norm(v2)
    if np.allclose(v1, v2):
        return np.eye(3)
    if np.allclose(v1, -v2):
        return 2 * np.outer(v1, v1) - np.eye(3)
    v = np.cross(v1, v2)
    s = np.linalg.norm(v)
    c = np.dot(v1, v2)
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + vx + vx @ vx * ((1 - c) / (s ** 2))


def rotate_molecule(molecule, anchor_idx, molecule_type, direction_vector=None, normal_vector=None):
    molecule = molecule.copy()
    anchor_pos = molecule.get_positions()[anchor_idx]
    positions = molecule.get_positions() - anchor_pos
    molecule.set_positions(positions)
    if molecule_type == 'H2' and direction_vector is not None:
        rotation = rotation_matrix_from_vectors(np.array([1, 0, 0]), np.array(direction_vector))
    elif molecule_type == 'H2O' and normal_vector is not None:
        rotation = rotation_matrix_from_vectors(np.array([0, 0, 1]), np.array(normal_vector))
    else:
        rotation = np.eye(3)
    positions = molecule.get_positions()
    molecule.set_positions(positions @ rotation.T)
    return molecule


def create_single_adsorption(surface_atoms, molecule_type, adsorption_site, adsorption_distance,
                             direction_vector=None, normal_vector=None):
    """单吸附结构（与单吸附 notebook 一致）"""
    site_pos = get_surface_site_position(surface_atoms, site=adsorption_site)
    molecule, anchor_idx = create_molecule(molecule_type)
    if molecule_type == 'H2':
        molecule = rotate_molecule(molecule, anchor_idx, molecule_type, direction_vector=direction_vector)
    elif molecule_type == 'H2O':
        molecule = rotate_molecule(molecule, anchor_idx, molecule_type, normal_vector=normal_vector)
    mol_positions = molecule.get_positions()
    if molecule_type in ['H2O', 'H2']:
        mol_center = np.mean(mol_positions, axis=0)
    else:
        mol_center = mol_positions[anchor_idx]
    target_pos = site_pos.copy()
    target_pos[2] += adsorption_distance
    anchor_pos = mol_positions[anchor_idx]
    if molecule_type in ['H2O', 'H2']:
        translation = target_pos - mol_center
    else:
        translation = target_pos - anchor_pos
    mol_positions = molecule.get_positions() + translation
    molecule.set_positions(mol_positions)
    return molecule

In [ ]:
# ==========================================================
# NEB 原子对齐：重排 fin 的吸附原子，使总位移最小（最短路径）
# ==========================================================

def align_fin_for_neb(ini_atoms, fin_atoms, n_fixed=48):
    """
    重排 fin 的吸附原子顺序，使 ini[i] 与 fin[i] 的位移总和最小。
    用于 nebmake.pl 前，确保线性插值时每个原子走最短路径到目标位点。
    
    参数:
        ini_atoms, fin_atoms: ASE Atoms（原子数、元素种类相同）
        n_fixed: 前 n_fixed 个原子为基底（Cu），不重排
    
    返回:
        fin_aligned: 重排后的 fin（元素顺序与 ini 一致，位置为最优匹配）
    """
    from itertools import permutations
    from collections import defaultdict
    ini = ini_atoms.copy()
    fin = fin_atoms.copy()
    n = len(ini)
    if len(fin) != n:
        raise ValueError(f"原子数不一致: ini={n}, fin={len(fin)}")
    ini_pos = ini.get_positions()
    fin_pos = fin.get_positions()
    ini_sym = ini.get_chemical_symbols()
    fin_sym = fin.get_chemical_symbols()
    
    # 按元素分组
    ini_elements = defaultdict(list)
    fin_elements = defaultdict(list)
    for i in range(n_fixed, n):
        ini_elements[ini_sym[i]].append(i)
        fin_elements[fin_sym[i]].append(i)
    
    new_pos = np.zeros_like(ini_pos)
    new_pos[:n_fixed] = fin_pos[:n_fixed]  # 基底不变
    
    for elem, ini_idxs in ini_elements.items():
        fin_idxs = fin_elements.get(elem, [])
        if len(ini_idxs) != len(fin_idxs):
            for idx in ini_idxs:
                new_pos[idx] = fin_pos[idx]
            continue
        if len(ini_idxs) == 1:
            new_pos[ini_idxs[0]] = fin_pos[fin_idxs[0]]
            continue
        best_sum, best_perm = np.inf, list(range(len(fin_idxs)))
        for p in permutations(range(len(fin_idxs))):
            s = sum(np.linalg.norm(ini_pos[ini_idxs[j]] - fin_pos[fin_idxs[p[j]]])**2 for j in range(len(ini_idxs)))
            if s < best_sum:
                best_sum, best_perm = s, list(p)
        for j, idx in enumerate(ini_idxs):
            new_pos[idx] = fin_pos[fin_idxs[best_perm[j]]]
    
    from ase import Atoms
    fin_aligned = Atoms(ini_sym, positions=new_pos, cell=fin.get_cell(), pbc=fin.get_pbc())
    if hasattr(fin, 'get_constraints') and fin.get_constraints():
        fin_aligned.set_constraint(fin.get_constraints())
    return fin_aligned


print("✓ align_fin_for_neb 已定义（NEB 前调用可最小化插值位移）")

## NEB 原子对齐用法

在运行 `nebmake.pl` 之前，对 fin 调用 `align_fin_for_neb`，可最小化插值位移：

```python
# 读取 ini、fin（如 CONTCAR）
ini_atoms = read("ini/CONTCAR", format='vasp')
fin_atoms = read("fin/CONTCAR", format='vasp')
fin_aligned = align_fin_for_neb(ini_atoms, fin_atoms, n_fixed=48)
write("fin/CONTCAR", fin_aligned, format='vasp', vasp5=True)
# 然后再运行 nebmake.pl
```

In [ ]:
# ==========================================================
# 双位点专用：确定性位点对 + 双吸附结构
# ==========================================================

def get_surface_site_pair(surface_atoms, site_A, site_B, shortest_path=True):
    """
    获取双位点吸附的 A、B 位点坐标（高对称位点，可选最短路径）
    
    规则：
    - A 位点：晶胞中心对应的 site_A 类型
    - B 位点：shortest_path=True 时取**最近邻**高对称位点（距离 a/4≈1.8 Å）；
              False 时取对角 (a/4, b/4)（距离约 2.6 Å）
    """
    cell = surface_atoms.get_cell()
    a = np.linalg.norm(cell[0])
    b = np.linalg.norm(cell[1])
    pos_A = get_surface_site_position(surface_atoms, site=site_A)
    cu_indices = [i for i, s in enumerate(surface_atoms.get_chemical_symbols()) if s.upper() == 'CU']
    cu_positions = surface_atoms.get_positions()[cu_indices]
    top_z = np.max(cu_positions[:, 2])
    top_cu = cu_positions[cu_positions[:, 2] > top_z - 0.1]
    
    if site_A == site_B:
        # shortest_path 为函数参数，默认 True
        if shortest_path:
            b_ref_xy = np.array([a/2 + a/4, b/2])  # 最近邻，距离 a/4≈1.8 Å
        else:
            b_ref_xy = np.array([a/4, b/4])  # 对角，距离约 2.6 Å
        if site_B == 'hollow':
            pos_B = np.array([b_ref_xy[0], b_ref_xy[1], top_z])
        elif site_B == 'top':
            dist = np.linalg.norm(top_cu[:, :2] - b_ref_xy, axis=1)
            nearest = top_cu[np.argmin(dist)]
            pos_B = np.array([nearest[0], nearest[1], top_z])
        elif site_B == 'bridge':
            near_pt = top_cu[np.argmin(np.linalg.norm(top_cu[:, :2] - b_ref_xy, axis=1))]
            dist_to_near = np.linalg.norm(top_cu[:, :2] - near_pt[:2], axis=1)
            dist_to_near[np.argmin(np.linalg.norm(top_cu[:, :2] - b_ref_xy, axis=1))] = np.inf
            second = top_cu[np.argmin(np.abs(dist_to_near - a/2))]
            pos_B = (near_pt + second) / 2
            pos_B[2] = top_z
    else:
        pos_B = get_surface_site_position(surface_atoms, site=site_B)
        # 若 A、B 同类型不同位点，pos_B 已与 pos_A 不同
        if np.linalg.norm(pos_A[:2] - pos_B[:2]) < 0.5:
            # 若意外重叠，将 B 移到 (a/4, b/4) 附近
            cu_indices = [i for i, s in enumerate(surface_atoms.get_chemical_symbols()) if s.upper() == 'CU']
            cu_positions = surface_atoms.get_positions()[cu_indices]
            top_z = np.max(cu_positions[:, 2])
            pos_B = get_surface_site_position(surface_atoms, site=site_B)
            # 尝试用 a/4,b/4 作为偏移参考
            pos_B = np.array([a/4, b/4, top_z])  # 简单 fallback
    
    return pos_A, pos_B


def create_double_adsorption_structure(surface_atoms,
                                        mol_A_type, site_A,
                                        mol_B_type, site_B,
                                        adsorption_distance=2.5,
                                        shortest_path=True,
                                        mol_A_direction=None, mol_A_normal=None,
                                        mol_B_direction=None, mol_B_normal=None):
    """
    创建双位点吸附结构。shortest_path=True 时 B 取最近邻高对称位点。
    原子顺序：Cu → A 物种原子 → B 物种原子
    """
    pos_A, pos_B = get_surface_site_pair(surface_atoms, site_A, site_B, shortest_path=shortest_path)
    
    mol_A, _ = create_single_adsorption(
        surface_atoms, mol_A_type, site_A, adsorption_distance,
        direction_vector=mol_A_direction, normal_vector=mol_A_normal
    )
    # 覆盖 A 的 z 高度为 pos_A + dist
    mol_A_pos = mol_A.get_positions()
    if mol_A_type in ['H2O', 'H2']:
        cen_A = np.mean(mol_A_pos, axis=0)
    else:
        cen_A = mol_A_pos[0]
    shift_A = (pos_A + np.array([0, 0, adsorption_distance])) - cen_A
    mol_A.set_positions(mol_A_pos + shift_A)
    
    mol_B, anchor_B = create_molecule(mol_B_type)
    if mol_B_type == 'H2':
        mol_B = rotate_molecule(mol_B, anchor_B, mol_B_type, direction_vector=mol_B_direction)
    elif mol_B_type == 'H2O':
        mol_B = rotate_molecule(mol_B, anchor_B, mol_B_type, normal_vector=mol_B_normal)
    mol_B_pos = mol_B.get_positions()
    if mol_B_type in ['H2O', 'H2']:
        cen_B = np.mean(mol_B_pos, axis=0)
    else:
        cen_B = mol_B_pos[anchor_B]
    target_B = pos_B.copy()
    target_B[2] += adsorption_distance
    mol_B.set_positions(mol_B_pos + (target_B - cen_B))
    
    # 合并：表面 + A + B（固定顺序）
    final = surface_atoms.copy()
    final.extend(mol_A)
    final.extend(mol_B)
    return final


print("✓ 双位点核心函数已定义")

## 第三步：可视化双位点位置（可选）

In [ ]:
def visualize_double_sites(surface_atoms, site_A='hollow', site_B='hollow', shortest_path=True):
    pos_A, pos_B = get_surface_site_pair(surface_atoms, site_A, site_B, shortest_path=shortest_path)
    cell = surface_atoms.get_cell()
    a = np.linalg.norm(cell[0])
    b = np.linalg.norm(cell[1])
    center = np.array([a/2, b/2])
    dist_AB = np.linalg.norm(pos_A[:2] - pos_B[:2])
    print(f"\n{'='*60}")
    print(f"双位点 A({site_A}) - B({site_B})")
    print(f"A: ({pos_A[0]:.4f}, {pos_A[1]:.4f}, {pos_A[2]:.4f}) Å")
    print(f"B: ({pos_B[0]:.4f}, {pos_B[1]:.4f}, {pos_B[2]:.4f}) Å")
    print(f"A-B 平面距离: {dist_AB:.4f} Å")
    print(f"{'='*60}\n")
    return pos_A, pos_B

print('--- shortest_path=True（最近邻，默认）---')
visualize_double_sites(surface_atoms, 'hollow', 'hollow', shortest_path=True)
print('--- shortest_path=False（对角）---')
visualize_double_sites(surface_atoms, 'hollow', 'hollow', shortest_path=False)

## 第四步：生成双位点结构

命名规则：`job_POSCAR_A_{site_A}_{mol_A}__B_{site_B}_{mol_B}`（与 GO 目录一致）

In [ ]:
# ==========================================================
# 双位点吸附：H @ hollow + H @ hollow（2H → H2 的 NEB 初态）
# ==========================================================

mol_A_type = 'H'
site_A = 'hollow'
mol_B_type = 'H'
site_B = 'hollow'
adsorption_distance = 2.5

final_atoms = create_double_adsorption_structure(
    surface_atoms, mol_A_type, site_A, mol_B_type, site_B,
    adsorption_distance=adsorption_distance
)

job_name = f"job_POSCAR_A_{site_A}_{mol_A_type}__B_{site_B}_{mol_B_type}"
poscar_name = f"POSCAR_A_{site_A}_{mol_A_type}__B_{site_B}_{mol_B_type}.vasp"
folder_name = f"{site_A}_{mol_A_type}_{site_B}_{mol_B_type}"

out_dir = base_output_dir / folder_name
out_dir.mkdir(exist_ok=True)
job_dir = out_dir / job_name
job_dir.mkdir(exist_ok=True)

write(str(job_dir / "POSCAR"), final_atoms, format='vasp', vasp5=True)
write(str(out_dir / poscar_name), final_atoms, format='vasp', vasp5=True)

print(f"✓ 生成: {job_name}")
print(f"  原子数: {len(final_atoms)} (Cu: 48, {mol_A_type}: {final_atoms.get_chemical_symbols().count(mol_A_type)}, {mol_B_type}: {final_atoms.get_chemical_symbols().count(mol_B_type)})")
print(f"  输出: {job_dir}")

In [ ]:
# ==========================================================
# 双位点吸附：O @ hollow + H @ hollow（H_O）
# ==========================================================

mol_A_type = 'O'
site_A = 'hollow'
mol_B_type = 'H'
site_B = 'hollow'
adsorption_distance = 2.5

final_atoms = create_double_adsorption_structure(
    surface_atoms, mol_A_type, site_A, mol_B_type, site_B,
    adsorption_distance=adsorption_distance
)

job_name = f"job_POSCAR_A_{site_A}_{mol_A_type}__B_{site_B}_{mol_B_type}"
poscar_name = f"POSCAR_A_{site_A}_{mol_A_type}__B_{site_B}_{mol_B_type}.vasp"
folder_name = f"{site_A}_{mol_A_type}_{site_B}_{mol_B_type}"

out_dir = base_output_dir / folder_name
out_dir.mkdir(exist_ok=True)
job_dir = out_dir / job_name
job_dir.mkdir(exist_ok=True)

write(str(job_dir / "POSCAR"), final_atoms, format='vasp', vasp5=True)
write(str(out_dir / poscar_name), final_atoms, format='vasp', vasp5=True)

print(f"✓ 生成: {job_name}")
print(f"  原子数: {len(final_atoms)}")
print(f"  输出: {job_dir}")

In [ ]:
# ==========================================================
# 双位点吸附：OH @ hollow + H @ hollow（OH_H）
# ==========================================================

mol_A_type = 'OH'
site_A = 'hollow'
mol_B_type = 'H'
site_B = 'hollow'
adsorption_distance = 2.5

final_atoms = create_double_adsorption_structure(
    surface_atoms, mol_A_type, site_A, mol_B_type, site_B,
    adsorption_distance=adsorption_distance
)

job_name = f"job_POSCAR_A_{site_A}_{mol_A_type}__B_{site_B}_{mol_B_type}"
poscar_name = f"POSCAR_A_{site_A}_{mol_A_type}__B_{site_B}_{mol_B_type}.vasp"
folder_name = f"{site_A}_{mol_A_type}_{site_B}_{mol_B_type}"

out_dir = base_output_dir / folder_name
out_dir.mkdir(exist_ok=True)
job_dir = out_dir / job_name
job_dir.mkdir(exist_ok=True)

write(str(job_dir / "POSCAR"), final_atoms, format='vasp', vasp5=True)
write(str(out_dir / poscar_name), final_atoms, format='vasp', vasp5=True)

print(f"✓ 生成: {job_name}")
print(f"  原子数: {len(final_atoms)}")
print(f"  输出: {job_dir}")

## 说明：NEB 无扭转检查清单

1. **原子顺序一致**：ini 和 fin 的 POSCAR 中，原子顺序必须一致（Cu → A 原子 → B 原子）
2. **位点可重复**：同一 (site_A, site_B) 组合每次生成的坐标相同
3. **元素顺序**：VASP 中按元素组排列，与 `create_double_adsorption_structure` 输出的顺序一致
4. **nebmake.pl**：使用 `nebmake.pl ini/CONTCAR fin/CONTCAR N` 时，两文件原子数、元素顺序一致即可线性插值